In [9]:
%pip install scikit-learn pandas numpy


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd

TRAIN_DATA = pd.read_csv('train-data.csv', index_col='id')
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA = pd.read_csv('test-data.csv', index_col='id')

In [11]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

if 'subscription' in TRAIN_DATA.columns:
    TRAIN_DATA = TRAIN_DATA.drop(columns=['subscription'])

def add_features(df):
    df = df.copy()
    # Previous campaign features
    df['contacted_recently'] = ((df['pdays'] != -1) & (df['pdays'] < 30)).astype(int)
    df['prev_success'] = (df['poutcome'] == 'SUC').astype(int)
    df['never_contacted'] = (df['previous'] == 0).astype(int)
    # Call quality features
    df['long_call'] = (df['duration'] > 300).astype(int)
    df['very_long_call'] = (df['duration'] > 600).astype(int)
    df['short_call'] = (df['duration'] < 60).astype(int)
    # Financial features
    df['has_balance'] = (df['balance'] > 0).astype(int)
    df['high_balance'] = (df['balance'] > 1000).astype(int)
    df['debt'] = (df['balance'] < 0).astype(int)
    # Campaign pressure features
    df['over_contacted'] = (df['campaign'] > 5).astype(int)
    df['first_contact'] = (df['campaign'] == 1).astype(int)
    # Age groups
    df['is_young'] = (df['age'] < 30).astype(int)
    df['is_retired_age'] = (df['age'] > 60).astype(int)
    df['is_middle_age'] = ((df['age'] >= 30) & (df['age'] <= 60)).astype(int)
    # Interaction features
    df['long_call_prev_success'] = df['long_call'] * df['prev_success']
    df['long_call_never_contacted'] = df['long_call'] * df['never_contacted']
    df['high_balance_long_call'] = df['high_balance'] * df['long_call']
    # Add these INSIDE add_features, after existing features

    # Duration buckets (more granular than long/short)
    df['very_short_call'] = (df['duration'] < 30).astype(int)
    df['medium_call'] = ((df['duration'] >= 60) & (df['duration'] <= 300)).astype(int)

    # Balance buckets
    df['very_high_balance'] = (df['balance'] > 5000).astype(int)
    df['medium_balance'] = ((df['balance'] > 0) & (df['balance'] <= 1000)).astype(int)

    # Previous contact success rate signal
    df['pdays_recent'] = df['pdays'].apply(lambda x: 0 if x == -1 else x)
    df['multiple_prev_contacts'] = (df['previous'] > 2).astype(int)

    # Month groupings (Q1/Q2/Q3/Q4 tend to differ)
    df['q1'] = df['month'].isin([1, 2, 3]).astype(int)
    df['q2'] = df['month'].isin([4, 5, 6]).astype(int)
    df['q3'] = df['month'].isin([7, 8, 9]).astype(int)
    df['q4'] = df['month'].isin([10, 11, 12]).astype(int)

    # Strong combined signals
    df['success_signal'] = (
        (df['duration'] > 300) & 
        (df['poutcome'] == 'SUC')
    ).astype(int)

    df['warm_lead'] = (
        (df['contacted_recently'] == 1) & 
        (df['prev_success'] == 1)
    ).astype(int)

    df['cold_lead'] = (
        (df['never_contacted'] == 1) & 
        (df['short_call'] == 1)
    ).astype(int)
    return df

TRAIN_DATA = add_features(TRAIN_DATA)
TEST_DATA = add_features(TEST_DATA)

cat_cols = ['job', 'marital_status', 'education', 'default_loan',
            'housing_loan', 'personal_loan', 'contact_type', 'poutcome']
num_cols = ['age', 'balance', 'day', 'month', 'duration',
            'campaign', 'pdays', 'previous',
            'contacted_recently', 'prev_success', 'long_call',
            'never_contacted', 'very_long_call', 'short_call',
            'has_balance', 'high_balance', 'debt',
            'over_contacted', 'first_contact',
            'is_young', 'is_retired_age', 'is_middle_age',
            'long_call_prev_success', 'long_call_never_contacted',
            'high_balance_long_call',
            # NEW ones below
            'very_short_call', 'medium_call',
            'very_high_balance', 'medium_balance',
            'pdays_recent', 'multiple_prev_contacts',
            'q1', 'q2', 'q3', 'q4',
            'success_signal', 'warm_lead', 'cold_lead']

def preprocess(df, encoder):
    df = df.copy()
    # OrdinalEncoder for categoricals - trees handle this fine
    cat_enc = pd.DataFrame(
        encoder.transform(df[cat_cols]),
        columns=cat_cols,
        index=df.index
    )
    num_df = df[num_cols].reset_index(drop=True)
    cat_enc = cat_enc.reset_index(drop=True)
    return pd.concat([cat_enc, num_df], axis=1)

# Trees don't need scaling or polynomial features!
ENCODER = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1).fit(TRAIN_DATA[cat_cols])

X_train = preprocess(TRAIN_DATA, ENCODER)
X_test = preprocess(TEST_DATA, ENCODER)
y_train = TRAIN_LABEL['subscription'].values

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)

X_train shape: (29839, 46)
X_test shape: (19893, 46)


In [12]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.ensemble import HistGradientBoostingClassifier

rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

configs = [
    {'learning_rate': 0.05, 'max_depth': 6, 'max_iter': 500, 'l2_regularization': 0.1},
    {'learning_rate': 0.05, 'max_depth': 6, 'max_iter': 500, 'l2_regularization': 0.5},
    {'learning_rate': 0.05, 'max_depth': 6, 'max_iter': 500, 'l2_regularization': 1.0},
    {'learning_rate': 0.05, 'max_leaf_nodes': 31,  'max_iter': 500},
    {'learning_rate': 0.05, 'max_leaf_nodes': 63,  'max_iter': 500},
    {'learning_rate': 0.05, 'max_leaf_nodes': 127, 'max_iter': 500},
    {'learning_rate': 0.02, 'max_leaf_nodes': 63,  'max_iter': 1000, 'l2_regularization': 0.1},
    {'learning_rate': 0.02, 'max_leaf_nodes': 127, 'max_iter': 1000, 'l2_regularization': 0.1},
]

best_score = 0
best_config = None

for cfg in configs:
    m = HistGradientBoostingClassifier(
        class_weight='balanced',
        random_state=1,
        **cfg
    )
    scores = cross_val_score(m, X_train, y_train, cv=rskf, scoring='balanced_accuracy', n_jobs=-1)
    print(f'{cfg} → CV = {scores.mean():.4f} ± {scores.std():.4f}')
    if scores.mean() > best_score:
        best_score = scores.mean()
        best_config = cfg

print(f'\n🏆 Best config: {best_config}')
print(f'🏆 Best CV: {best_score:.4f}')

{'learning_rate': 0.05, 'max_depth': 6, 'max_iter': 500, 'l2_regularization': 0.1} → CV = 0.8625 ± 0.0053
{'learning_rate': 0.05, 'max_depth': 6, 'max_iter': 500, 'l2_regularization': 0.5} → CV = 0.8623 ± 0.0053
{'learning_rate': 0.05, 'max_depth': 6, 'max_iter': 500, 'l2_regularization': 1.0} → CV = 0.8642 ± 0.0054
{'learning_rate': 0.05, 'max_leaf_nodes': 31, 'max_iter': 500} → CV = 0.8653 ± 0.0066
{'learning_rate': 0.05, 'max_leaf_nodes': 63, 'max_iter': 500} → CV = 0.8628 ± 0.0072
{'learning_rate': 0.05, 'max_leaf_nodes': 127, 'max_iter': 500} → CV = 0.8569 ± 0.0073
{'learning_rate': 0.02, 'max_leaf_nodes': 63, 'max_iter': 1000, 'l2_regularization': 0.1} → CV = 0.8619 ± 0.0057
{'learning_rate': 0.02, 'max_leaf_nodes': 127, 'max_iter': 1000, 'l2_regularization': 0.1} → CV = 0.8561 ± 0.0070

🏆 Best config: {'learning_rate': 0.05, 'max_leaf_nodes': 31, 'max_iter': 500}
🏆 Best CV: 0.8653


In [13]:
model = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_depth=6,
    max_leaf_nodes=31,
    max_iter=500,
    class_weight='balanced',
    random_state=1
)

cv_scores = cross_val_score(model, X_train, y_train, cv=rskf, scoring='balanced_accuracy', n_jobs=-1)
print(f'CV Balanced Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Per-fold scores: {cv_scores}')

CV Balanced Accuracy: 0.8632 ± 0.0053
Per-fold scores: [0.8590942  0.86507016 0.86208816 0.87043056 0.86635581 0.86729341
 0.86177392 0.86389034 0.85029741 0.86534688 0.85999542 0.86408605
 0.87355409 0.86066289 0.85848105]


In [14]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import balanced_accuracy_score

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
I_train, I_val = next(splitter.split(X_train, y_train))
X_tr, X_val = X_train.iloc[I_train], X_train.iloc[I_val]
y_tr, y_val = y_train[I_train], y_train[I_val]

model.fit(X_tr, y_tr)
val_probs = model.predict_proba(X_val)[:, 1]

best_threshold, best_ba = 0.5, 0.0
for thresh in np.arange(0.1, 0.9, 0.001):
    preds = (val_probs >= thresh).astype(int)
    ba = balanced_accuracy_score(y_val, preds)
    if ba > best_ba:
        best_ba = ba
        best_threshold = thresh

print(f'Best threshold: {best_threshold:.3f}')
print(f'Best Val Balanced Accuracy: {best_ba:.4f}')

Best threshold: 0.459
Best Val Balanced Accuracy: 0.8788


In [15]:
final_model = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_depth=6,
    max_leaf_nodes=31,
    max_iter=500,
    class_weight='balanced',
    random_state=1
)

final_model.fit(X_train, y_train)
test_probs = final_model.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= best_threshold).astype(int)
print(f'Prediction distribution — 0: {(test_preds==0).sum()}, 1: {(test_preds==1).sum()}')

Prediction distribution — 0: 15229, 1: 4664


In [16]:
submission = pd.DataFrame({
    'id': TEST_DATA.index,
    'subscription': test_preds
})

submission.to_csv('submission33r.csv', index=False)
print('Saved! Preview:')
print(submission.head())

Saved! Preview:
      id  subscription
0  37797             0
1  37798             0
2  37799             0
3  37800             0
4  37801             1
